# Базовая проверка SmolVLM-Instruct на GQA-ru

В этом ноутбуке проверяется, как локальная `SmolVLM-Instruct` решает русскоязычный VQA-датасет GQA-ru без дообучения.

Идея простая: берем небольшую подвыборку, получаем предсказания модели, считаем точное совпадение / точность и смотрим, где модель ошибается.

## Как запустить

Перед запуском проверь, что в корне проекта есть такие папки:

- `models/SmolVLM-Instruct` — локальная базовая модель SmolVLM;
- `data/GQA-ru` — локальный датасет GQA-ru;
- `outputs/` — создастся автоматически для результатов базовой проверки.

Нужные зависимости:

```bash
pip install torch "transformers>=4.46.0" accelerate pandas pyarrow pillow tqdm plotly
```

Что делает ноутбук:

- автоматически смотрит структуру `data/GQA-ru`;
- выбирает `testdev`, если он есть;
- запускает получение предсказаний на небольшой подвыборке;
- считает метрики и сохраняет результаты локально.

Что ноутбук не делает:

- не обучает модель;
- не запускает LoRA/QLoRA;
- не использует MLflow/DagsHub;
- не трогает файлы в `data/` и `models/`.

## 1. Импорты и настройки

В начале держим все важные настройки в одном месте. По умолчанию запускаем только 100 примеров, чтобы не перегружать GPU и быстро получить первую оценку базовой модели.

In [ ]:
from pathlib import Path
from io import BytesIO
import json
import re
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
from tqdm.auto import tqdm

import torch

try:
    from transformers import AutoProcessor

    try:
        from transformers import AutoModelForImageTextToText as SmolVLMModelClass
    except ImportError:
        try:
            from transformers import AutoModelForVision2Seq as SmolVLMModelClass
        except ImportError:
            from transformers import Idefics3ForConditionalGeneration as SmolVLMModelClass
except ImportError as error:
    raise ImportError(
        "Не найден transformers с поддержкой SmolVLM/Idefics3. "
        "Установи или обнови пакет: pip install -U 'transformers>=4.46.0' accelerate"
    ) from error

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

px.defaults.template = "plotly_white"
pd.set_option("display.max_colwidth", 120)

DISPLAY_COLUMN_NAMES = {
    "name": "Параметр",
    "value": "Значение",
    "folder": "Папка",
    "split": "Сплит",
    "role": "Тип",
    "num_files": "Файлов",
    "num_rows_first_file": "Строк в первом файле",
    "columns": "Колонки",
    "example_file": "Пример файла",
    "table": "Таблица",
    "rows": "Строки",
    "field": "Поле",
    "column": "Колонка",
    "_id": "ID вопроса",
    "_image_id": "ID изображения",
    "_question": "Вопрос",
    "_answer": "Правильный ответ",
    "row_id": "ID строки",
    "id": "ID вопроса",
    "image_id": "ID изображения",
    "question": "Вопрос",
    "ground_truth": "Правильный ответ",
    "prediction": "Предсказание модели",
    "ground_truth_norm": "Правильный ответ (норм.)",
    "prediction_norm": "Предсказание модели (норм.)",
    "is_correct": "Верно",
    "is_yes_no": "Вопрос да/нет",
    "metric": "Метрика",
    "accuracy": "Точность",
    "n": "Количество",
    "num_errors": "Количество ошибок",
    "prediction_yes_no": "Предсказание да/нет",
    "count": "Количество",
    "artifact": "Артефакт",
    "path": "Путь",
    "status": "Статус",
    "prompt": "Промпт",
    "normalized_ground_truth": "Правильный ответ после нормализации",
    "cuda_available": "CUDA доступна",
    "gpu": "GPU",
    "inference_input_device": "Устройство для получения предсказаний",
    "model_class": "Класс модели",
    "message": "Сообщение",
}

DISPLAY_VALUE_REPLACEMENTS = {
    True: "да",
    False: "нет",
    "images": "изображения",
    "instructions": "инструкции",
    "unknown": "неизвестно",
    "overall": "общая",
    "yes/no": "да/нет",
    "non-yes/no": "не да/нет",
    "yes": "да",
    "no": "нет",
    "other": "другое",
    "correct": "верно",
    "incorrect": "неверно",
    "predictions": "предсказания",
    "metrics": "метрики",
    "error_examples": "примеры ошибок",
    "question": "вопрос",
    "answer": "правильный ответ",
    "instruction_id": "ID вопроса",
    "image_id": "ID изображения",
    "image_table_id": "ID изображения в таблице",
    "image": "изображение",
    "selected_split": "выбранный сплит",
    "available_examples": "доступно примеров",
    "baseline_sample_size": "размер подвыборки",
    "random_seed": "зерно случайности",
    "project_root": "корень проекта",
    "model_dir": "папка модели",
    "data_dir": "папка датасета",
    "output_dir": "папка результатов",
}


from IPython.display import display

def show_df(df, max_rows=20, index=False):
    table = df.head(max_rows).copy() if max_rows is not None else df.copy()

    if len(table) == 0:
        display(pd.DataFrame({"message": ["empty table"]}))
        return

    if not index:
        table = table.reset_index(drop=True)

    display(
        table.style
        .set_properties(**{
            "text-align": "left",
            "white-space": "pre-wrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
            ]},
            {"selector": "td", "props": [
                ("vertical-align", "top"),
            ]},
        ])
    )

In [ ]:
# Основные настройки базовой проверки.
SAMPLE_SIZE = 100
RANDOM_SEED = 42
MAX_NEW_TOKENS = 16

# Поддерживаем запуск и из корня проекта, и из папки notebooks.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models" / "SmolVLM-Instruct"
DATA_DIR = PROJECT_ROOT / "data" / "GQA-ru"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "baseline_smolvlm_gqa"

config_df = pd.DataFrame([
    {"name": "project_root", "value": str(PROJECT_ROOT)},
    {"name": "model_dir", "value": str(MODEL_DIR)},
    {"name": "data_dir", "value": str(DATA_DIR)},
    {"name": "output_dir", "value": str(OUTPUT_DIR)},
    {"name": "cuda_available", "value": torch.cuda.is_available()},
])
show_df(config_df)

## 2. Структура датасета

Сначала не предполагаем схему датасета заранее. Посмотрим, какие parquet-папки есть внутри `data/GQA-ru`, сколько в них строк и какие колонки.

In [ ]:
def infer_split_name(folder_name):
    name = folder_name.lower()
    if "testdev" in name:
        return "testdev"
    if "validation" in name or "valid" in name:
        return "validation"
    if re.search(r"(^|_)val($|_)", name):
        return "validation"
    if "test" in name:
        return "test"
    if "train" in name:
        return "train"
    return folder_name


def infer_folder_role(folder_name):
    name = folder_name.lower()
    if "image" in name:
        return "images"
    if "instruction" in name or "question" in name or "annotation" in name:
        return "instructions"
    return "unknown"


def discover_parquet_folders(data_dir):
    rows = []
    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir() or folder.name.startswith("."):
            continue

        parquet_files = sorted(folder.glob("*.parquet"))
        if not parquet_files:
            continue

        first_file = parquet_files[0]
        parquet_file = pq.ParquetFile(first_file)
        rows.append({
            "folder": folder.name,
            "split": infer_split_name(folder.name),
            "role": infer_folder_role(folder.name),
            "num_files": len(parquet_files),
            "num_rows_first_file": parquet_file.metadata.num_rows,
            "columns": parquet_file.schema.names,
            "example_file": first_file.name,
            "path": folder,
        })
    return pd.DataFrame(rows)


structure_df = discover_parquet_folders(DATA_DIR)

show_structure = structure_df.drop(columns=["path"]).copy()
show_structure["columns"] = show_structure["columns"].apply(lambda cols: ", ".join(cols))
show_df(show_structure)

Теперь выведем 1-2 коротких примера из каждой найденной parquet-папки. Для изображений не читаем байты целиком, иначе ноутбук станет тяжелым и нечитаемым.

In [ ]:
def short_cell(value, max_len=140):
    if isinstance(value, dict):
        short = {}
        for key, item in value.items():
            if isinstance(item, (bytes, bytearray)):
                short[key] = f"<bytes: {len(item)} bytes>"
            else:
                short[key] = item
        return short
    if isinstance(value, (bytes, bytearray)):
        return f"<bytes: {len(value)} bytes>"
    text = str(value)
    if len(text) > max_len:
        return text[:max_len] + "..."
    return value


def shorten_dataframe(df):
    return df.applymap(short_cell)


def preview_parquet_folder(folder, n=2):
    files = sorted(Path(folder).glob("*.parquet"))
    if not files:
        return pd.DataFrame()

    parquet_file = pq.ParquetFile(files[0])
    all_columns = parquet_file.schema.names
    image_like_columns = {"image", "img", "picture"}
    read_columns = [col for col in all_columns if col.lower() not in image_like_columns]

    if read_columns:
        table = parquet_file.read_row_group(0, columns=read_columns).slice(0, n)
        preview_df = table.to_pandas()
    else:
        preview_df = pd.DataFrame(index=range(min(n, parquet_file.metadata.num_rows)))

    for col in all_columns:
        if col.lower() in image_like_columns:
            preview_df[col] = "<image bytes skipped in preview>"

    preview_df = preview_df[[col for col in all_columns if col in preview_df.columns]]
    return shorten_dataframe(preview_df)


for _, row in structure_df.iterrows():
    role_label = DISPLAY_VALUE_REPLACEMENTS.get(row["role"], row["role"])
    print(f"\n--- {row['folder']} | сплит={row['split']} | тип={role_label} ---")
    print("Колонки:", row["columns"])
    show_df(preview_parquet_folder(row["path"], n=2))



Датасет хранит инструкции и изображения в отдельных parquet-папках. Значит, перед получением предсказаний нужно явно соединить вопрос с изображением по `imageId`, иначе метрики будут считаться по неверным парам.

В структуре присутствует `testdev`, поэтому дальнейшая оценка может идти на сплите, который не используется для обучения. Это соответствует цели ноутбука: проверить качество SmolVLM без дообучения.

## 3. Выбор сплита для оценки

Для базовой проверки берем `testdev`, если он есть. Если нет, код пытается взять validation/test split. Это удобно, если структура датасета немного изменится.

In [ ]:
def choose_eval_folders(structure):
    if structure.empty:
        raise ValueError(f"No parquet folders found in {DATA_DIR}")

    split_priority = ["testdev", "validation", "test", "train"]

    for split in split_priority:
        split_rows = structure[structure["split"] == split]
        if split_rows.empty:
            continue

        instructions = split_rows[split_rows["role"] == "instructions"]
        images = split_rows[split_rows["role"] == "images"]

        if not instructions.empty:
            image_folder = None if images.empty else images.iloc[0]["path"]
            return split, instructions.iloc[0]["path"], image_folder

    # Fallback for unexpected folder names: take the first instruction-like folder.
    instructions = structure[structure["role"] == "instructions"]
    if not instructions.empty:
        split = instructions.iloc[0]["split"]
        images = structure[(structure["split"] == split) & (structure["role"] == "images")]
        image_folder = None if images.empty else images.iloc[0]["path"]
        return split, instructions.iloc[0]["path"], image_folder

    raise ValueError("Не удалось найти parquet-папку с инструкциями или вопросами.")


SELECTED_SPLIT, INSTRUCTIONS_DIR, IMAGES_DIR = choose_eval_folders(structure_df)

selected_split_df = pd.DataFrame([
    {"name": "selected_split", "value": SELECTED_SPLIT},
    {"name": "instructions_dir", "value": str(INSTRUCTIONS_DIR)},
    {"name": "images_dir", "value": str(IMAGES_DIR)},
])
show_df(selected_split_df)



Для оценки выбран `testdev`, поэтому базовая проверка не смешивается с train-частью датасета. Такой сплит удобен для честного сравнения с будущими LoRA/QLoRA экспериментами: предсказания до и после дообучения можно получать на одном и том же наборе примеров.

## 4. Загрузка сплита GQA-ru

Загружаем инструкции и изображения выбранного сплита. В GQA-ru они лежат отдельно, поэтому дальше аккуратно соединяем их по `imageId` / `id`.

In [ ]:
def read_parquet_folder(folder):
    files = sorted(Path(folder).glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"В папке не найдены parquet-файлы: {folder}")

    parts = [pd.read_parquet(file) for file in files]
    return pd.concat(parts, ignore_index=True)


instructions_df = read_parquet_folder(INSTRUCTIONS_DIR)
images_df = read_parquet_folder(IMAGES_DIR) if IMAGES_DIR is not None else pd.DataFrame()

load_summary_df = pd.DataFrame([
    {"table": "instructions", "rows": len(instructions_df), "columns": len(instructions_df.columns)},
    {"table": "images", "rows": len(images_df), "columns": len(images_df.columns)},
])
show_df(load_summary_df)

print("Колонки инструкций:", list(instructions_df.columns))
print("Колонки изображений:", list(images_df.columns))

print("\nПример инструкций:")
show_df(shorten_dataframe(instructions_df.head(2)))
if not images_df.empty:
    print("\nПример изображений:")
    show_df(shorten_dataframe(images_df.head(2)))

In [ ]:
def find_column(df, candidates, required=True):
    lower_to_original = {str(col).lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    if required:
        raise ValueError(f"Не найдена ни одна из колонок {candidates}. Доступные колонки: {list(df.columns)}")
    return None


QUESTION_COLUMN = find_column(instructions_df, ["question", "query", "prompt", "text"])
ANSWER_COLUMN = find_column(instructions_df, ["answer", "ground_truth", "gt_answer", "label"])
INSTRUCTION_ID_COLUMN = find_column(instructions_df, ["id", "question_id", "questionId"], required=False)
IMAGE_ID_COLUMN = find_column(instructions_df, ["imageId", "image_id", "imageid", "img_id"], required=False)

IMAGE_COLUMN = None
IMAGE_TABLE_ID_COLUMN = None

if not images_df.empty:
    IMAGE_TABLE_ID_COLUMN = find_column(images_df, ["id", "imageId", "image_id", "imageid"])
    IMAGE_COLUMN = find_column(images_df, ["image", "img", "picture"])
else:
    IMAGE_COLUMN = find_column(instructions_df, ["image", "img", "picture"], required=False)

column_mapping_df = pd.DataFrame([
    {"field": "question", "column": QUESTION_COLUMN},
    {"field": "answer", "column": ANSWER_COLUMN},
    {"field": "instruction_id", "column": INSTRUCTION_ID_COLUMN},
    {"field": "image_id", "column": IMAGE_ID_COLUMN},
    {"field": "image_table_id", "column": IMAGE_TABLE_ID_COLUMN},
    {"field": "image", "column": IMAGE_COLUMN},
])
show_df(column_mapping_df)

In [ ]:
def build_eval_dataframe(instructions, images):
    df = instructions.copy()

    if not images.empty and IMAGE_ID_COLUMN is not None:
        image_lookup = images.set_index(IMAGE_TABLE_ID_COLUMN)[IMAGE_COLUMN].to_dict()
        df["_image"] = df[IMAGE_ID_COLUMN].map(image_lookup)
    elif IMAGE_COLUMN is not None and IMAGE_COLUMN in df.columns:
        df["_image"] = df[IMAGE_COLUMN]
    else:
        raise ValueError("Не удалось найти изображения ни в таблице изображений, ни в таблице инструкций.")

    df["_question"] = df[QUESTION_COLUMN].astype(str).str.strip()
    df["_answer"] = df[ANSWER_COLUMN].astype(str).str.strip()

    if INSTRUCTION_ID_COLUMN is not None:
        df["_id"] = df[INSTRUCTION_ID_COLUMN].astype(str)
    else:
        df["_id"] = [str(i) for i in range(len(df))]

    if IMAGE_ID_COLUMN is not None:
        df["_image_id"] = df[IMAGE_ID_COLUMN].astype(str)
    else:
        df["_image_id"] = None

    df = df.dropna(subset=["_image", "_question", "_answer"]).reset_index(drop=True)
    return df


eval_df = build_eval_dataframe(instructions_df, images_df)

sample_n = len(eval_df) if SAMPLE_SIZE is None else min(SAMPLE_SIZE, len(eval_df))
eval_sample = eval_df.sample(n=sample_n, random_state=RANDOM_SEED).reset_index(drop=True)
eval_sample["_row_id"] = np.arange(len(eval_sample))

sample_summary_df = pd.DataFrame([
    {"name": "selected_split", "value": SELECTED_SPLIT},
    {"name": "available_examples", "value": len(eval_df)},
    {"name": "baseline_sample_size", "value": len(eval_sample)},
    {"name": "random_seed", "value": RANDOM_SEED},
])
show_df(sample_summary_df)
show_df(eval_sample[["_id", "_image_id", "_question", "_answer"]], max_rows=5)



В выбранном сплите вопросов больше, чем изображений: одно изображение используется для нескольких VQA-примеров. Поэтому единицей оценки является не картинка, а пара `изображение + вопрос`.

Подвыборка фиксируется через `SAMPLE_SIZE` и `RANDOM_SEED`. Это делает проверку быстрой и повторяемой, а при необходимости ту же логику можно запустить на полном `testdev`.

## 5. Проверка пары изображение-вопрос

Перед запуском модели проверяем одну пару `изображение + вопрос + правильный ответ`. Это простой контроль того, что соединение по `imageId` сработало корректно.

In [ ]:
def to_pil_image(image_value):
    if isinstance(image_value, Image.Image):
        return image_value.convert("RGB")

    if isinstance(image_value, dict):
        if image_value.get("bytes") is not None:
            return Image.open(BytesIO(image_value["bytes"])).convert("RGB")

        image_path = image_value.get("path")
        if image_path:
            image_path = Path(image_path)
            if not image_path.is_absolute():
                candidate_paths = [DATA_DIR / image_path, PROJECT_ROOT / image_path]
                image_path = next((path for path in candidate_paths if path.exists()), image_path)
            return Image.open(image_path).convert("RGB")

    if isinstance(image_value, (bytes, bytearray)):
        return Image.open(BytesIO(image_value)).convert("RGB")

    if isinstance(image_value, (str, Path)):
        image_path = Path(image_value)
        if not image_path.is_absolute():
            candidate_paths = [DATA_DIR / image_path, PROJECT_ROOT / image_path]
            image_path = next((path for path in candidate_paths if path.exists()), image_path)
        return Image.open(image_path).convert("RGB")

    raise TypeError(f"Unsupported image value type: {type(image_value)}")


first_example = eval_sample.iloc[0]
preview_image = to_pil_image(first_example["_image"])
fig = px.imshow(preview_image, title="Проверочное изображение")
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_layout(coloraxis_showscale=False, width=520, height=420)
fig.show()

sanity_df = pd.DataFrame([{
    "question": first_example["_question"],
    "ground_truth": first_example["_answer"],
}])
show_df(sanity_df)



Проверка нужна до запуска тяжелой модели: она подтверждает, что вопрос действительно относится к показанному изображению. Для VQA ошибка в связке изображения и вопроса сразу делает метрики бессмысленными.

## 6. Подготовка примеров и нормализация ответов

Промпт специально короткий: просим отвечать только по картинке и давать короткий ответ. Для метрик нормализуем ответы без тяжелой лемматизации: lower, strip, простая пунктуация и единый формат для `yes/no` и `да/нет`.

In [ ]:
PROMPT_TEMPLATE = """Answer the question using only the image. Give a short answer.
Question: {question}
Answer:"""


def prepare_example(row):
    question = str(row["_question"]).strip()
    answer = str(row["_answer"]).strip()
    image = to_pil_image(row["_image"])
    prompt = PROMPT_TEMPLATE.format(question=question)

    return {
        "id": str(row["_id"]),
        "image_id": None if pd.isna(row["_image_id"]) else str(row["_image_id"]),
        "image": image,
        "question": question,
        "answer": answer,
        "prompt": prompt,
    }


def normalize_answer(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    yes_no_aliases = {
        "yes": "yes",
        "y": "yes",
        "yeah": "yes",
        "yep": "yes",
        "true": "yes",
        "да": "yes",
        "ага": "yes",
        "верно": "yes",
        "no": "no",
        "n": "no",
        "nope": "no",
        "false": "no",
        "нет": "no",
        "неа": "no",
    }

    if text in yes_no_aliases:
        return yes_no_aliases[text]

    first_token = text.split()[0] if text else ""
    if first_token in yes_no_aliases:
        return yes_no_aliases[first_token]

    return text


prepared = prepare_example(eval_sample.iloc[0])
prompt_check_df = pd.DataFrame([{
    "prompt": prepared["prompt"],
    "normalized_ground_truth": normalize_answer(prepared["answer"]),
}])
show_df(prompt_check_df)



Нормализация специально не переводит ответы и не делает лемматизацию. Это сохраняет строгую метрику точного совпадения: если модель отвечает на английском вместо русского, ошибка остается видимой в метриках.

## 7. Загрузка SmolVLM

Загружаем модель строго из локальной папки. Если CUDA недоступна, ноутбук попробует CPU fallback, но на CPU получение предсказаний у VLM может быть очень медленным.

In [ ]:
device_check_df = pd.DataFrame([{
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}])
show_df(device_check_df)

In [ ]:
INFERENCE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def select_smolvlm_dtype():
    if INFERENCE_DEVICE != "cuda":
        return torch.float32

    is_bf16_supported = getattr(torch.cuda, "is_bf16_supported", lambda: False)
    return torch.bfloat16 if is_bf16_supported() else torch.float16


def load_smolvlm_model(model_dir):
    model_dtype = select_smolvlm_dtype()

    def _load(attn_implementation):
        model_kwargs = {
            "local_files_only": True,
            "_attn_implementation": attn_implementation,
        }
        try:
            return SmolVLMModelClass.from_pretrained(
                model_dir,
                torch_dtype=model_dtype,
                **model_kwargs,
            )
        except TypeError:
            # В новых версиях transformers может использоваться dtype вместо torch_dtype.
            return SmolVLMModelClass.from_pretrained(
                model_dir,
                dtype=model_dtype,
                **model_kwargs,
            )

    if INFERENCE_DEVICE == "cuda":
        try:
            model = _load("flash_attention_2")
            return model.to(INFERENCE_DEVICE), model_dtype, "flash_attention_2"
        except Exception as error:
            warnings.warn(
                "Не удалось загрузить SmolVLM с flash_attention_2. Пробуем eager. "
                f"Исходная ошибка: {error}"
            )

    model = _load("eager")
    return model.to(INFERENCE_DEVICE), model_dtype, "eager"


processor = AutoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
model, MODEL_DTYPE, ATTENTION_IMPLEMENTATION = load_smolvlm_model(MODEL_DIR)
model.eval()

model_info_df = pd.DataFrame([{
    "inference_input_device": INFERENCE_DEVICE,
    "model_class": model.__class__.__name__,
    "torch_dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "attention_implementation": ATTENTION_IMPLEMENTATION,
}])
show_df(model_info_df)

## 8. Получение предсказаний базовой модели

Генерация идет без sampling (`do_sample=False`), чтобы результат был воспроизводимым. Сохраняем не только ответ модели, но и нормализованные ответы для метрик.

In [ ]:
def generate_answer(image, question):
    prompt = PROMPT_TEMPLATE.format(question=question)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=text,
        images=[image],
        return_tensors="pt",
    )
    inputs = inputs.to(INFERENCE_DEVICE)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    output_text = re.sub(r"^\s*Assistant:\s*", "", output_text).strip()

    return output_text

In [ ]:
results = []

for _, row in tqdm(eval_sample.iterrows(), total=len(eval_sample)):
    example = prepare_example(row)
    prediction = generate_answer(example["image"], example["question"])

    gt_norm = normalize_answer(example["answer"])
    pred_norm = normalize_answer(prediction)

    results.append({
        "row_id": int(row["_row_id"]),
        "id": example["id"],
        "image_id": example["image_id"],
        "question": example["question"],
        "ground_truth": example["answer"],
        "prediction": prediction,
        "ground_truth_norm": gt_norm,
        "prediction_norm": pred_norm,
        "is_correct": gt_norm == pred_norm,
        "is_yes_no": gt_norm in {"yes", "no"},
        "_image": row["_image"],
    })

results_df = pd.DataFrame(results)
show_df(results_df.drop(columns=["_image"]).head())



Таблица после получения предсказаний показывает не только ответ модели, но и нормализованные версии правильного ответа и предсказания. Это делает ошибки интерпретируемыми: отдельно видны неверные объекты, английские ответы вместо русских и несовпадения после строгой нормализации.

## 9. Метрики

Главная метрика здесь — точное совпадение / точность после простой нормализации. Отдельно смотрим вопросы с ответом `да/нет` и остальные вопросы, потому что это обычно разные уровни сложности.

In [ ]:
def safe_accuracy(df):
    if len(df) == 0:
        return None
    return float(df["is_correct"].mean())


yes_no_df = results_df[results_df["is_yes_no"]].copy()
non_yes_no_df = results_df[~results_df["is_yes_no"]].copy()

metrics = {
    "model": str(MODEL_DIR),
    "dataset": str(DATA_DIR),
    "split": SELECTED_SPLIT,
    "sample_size": int(len(results_df)),
    "random_seed": RANDOM_SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "accuracy": safe_accuracy(results_df),
    "yes_no_accuracy": safe_accuracy(yes_no_df),
    "non_yes_no_accuracy": safe_accuracy(non_yes_no_df),
    "num_yes_no": int(len(yes_no_df)),
    "num_non_yes_no": int(len(non_yes_no_df)),
}

metrics_df = pd.DataFrame([
    {"metric": "overall", "accuracy": metrics["accuracy"], "n": len(results_df)},
    {"metric": "yes/no", "accuracy": metrics["yes_no_accuracy"], "n": len(yes_no_df)},
    {"metric": "non-yes/no", "accuracy": metrics["non_yes_no_accuracy"], "n": len(non_yes_no_df)},
])

show_df(metrics_df)



Общая точность остается низкой для готовой модели без дообучения. Разрыв между вопросами `да/нет` и открытыми короткими ответами показывает, что бинарные вопросы для SmolVLM в этом запуске существенно проще, чем генерация точного русскоязычного ответа.

Эта таблица — основная точка сравнения для будущего LoRA/QLoRA. Если дообучение улучшит только общий показатель, но не сократит разрыв по типам вопросов, качество модели останется неравномерным.

In [ ]:
errors_df = results_df[~results_df["is_correct"]].copy()
error_examples_df = errors_df[[
    "question",
    "ground_truth",
    "prediction",
    "ground_truth_norm",
    "prediction_norm",
]].head(20)

error_summary_df = pd.DataFrame([{"num_errors": len(errors_df)}])
show_df(error_summary_df)
show_df(error_examples_df)



Ошибки показывают два разных типа проблем. Часть предсказаний указывает на неверный объект или неверное пространственное отношение; другая часть фактически близка к правильному ответу, но дана на английском языке и не проходит строгую метрику точного совпадения.

Для базовой проверки это полезно: перед дообучением видно, что улучшать нужно не только визуальное понимание, но и стабильность русскоязычного формата ответа.

Если в подвыборке есть ответы `да/нет`, посмотрим матрицу ошибок. Она быстро показывает, путает ли модель `да` и `нет` или чаще уходит в другой ответ.

In [ ]:
if len(yes_no_df) > 0:
    confusion_df = yes_no_df.copy()
    confusion_df["prediction_yes_no"] = confusion_df["prediction_norm"].where(
        confusion_df["prediction_norm"].isin(["yes", "no"]),
        "other",
    )
    yes_no_confusion = pd.crosstab(
        confusion_df["ground_truth_norm"],
        confusion_df["prediction_yes_no"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    )
else:
    yes_no_confusion = pd.DataFrame()

yes_no_confusion = yes_no_confusion.rename_axis(
    index="Правильный ответ",
    columns="Предсказание модели",
)
show_df(yes_no_confusion, index=True)



Матрица ошибок отделяет бинарные ошибки от ошибок открытого VQA. В текущем запуске отрицательные ответы выглядят стабильнее, а среди положительных вопросов заметны ложные отрицания.

Такой срез важен для будущего дообучения: модель может улучшить общий показатель точного совпадения, но при этом сохранить перекос в сторону `нет`, если не контролировать вопросы `да/нет` отдельно.

## 10. Сохранение предсказаний и метрик

Сохраняем все локально в `outputs/baseline_smolvlm_gqa`. Эти файлы удобно сравнивать с будущим LoRA/QLoRA запуском.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_columns = [
    "id",
    "image_id",
    "question",
    "ground_truth",
    "prediction",
    "ground_truth_norm",
    "prediction_norm",
    "is_correct",
    "is_yes_no",
]

predictions_path = OUTPUT_DIR / "predictions.jsonl"
metrics_path = OUTPUT_DIR / "metrics.json"
errors_path = OUTPUT_DIR / "error_examples.csv"

records_df = results_df[prediction_columns].copy()
records_df["is_correct"] = records_df["is_correct"].astype(bool)
records_df["is_yes_no"] = records_df["is_yes_no"].astype(bool)

with predictions_path.open("w", encoding="utf-8") as file:
    for record in records_df.to_dict(orient="records"):
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

errors_df[prediction_columns].to_csv(errors_path, index=False, encoding="utf-8-sig")

saved_files_df = pd.DataFrame([
    {"artifact": "predictions", "path": str(predictions_path)},
    {"artifact": "metrics", "path": str(metrics_path)},
    {"artifact": "error_examples", "path": str(errors_path)},
])
show_df(saved_files_df)

## 11. Визуализации Plotly

Сделаем несколько простых графиков: общая точность, сравнение `да/нет` и остальных вопросов, а также самые частые правильные ответы среди ошибок модели.

In [ ]:
overall_plot_df = pd.DataFrame({
    "Метрика": ["Точное совпадение / точность"],
    "Точность": [metrics["accuracy"]],
})

fig = px.bar(
    overall_plot_df,
    x="Метрика",
    y="Точность",
    text=overall_plot_df["Точность"].map(lambda value: f"{value:.3f}"),
    title="Общая точность базовой модели",
)
fig.update_xaxes(title_text="Метрика")
fig.update_yaxes(title_text="Точность", range=[0, 1])
fig.update_traces(textposition="outside")
fig.show()



График фиксирует стартовое качество модели без дообучения. Для SmolVLM на GQA-ru это строгая проверка без дообучения: модель получает только изображение и вопрос, а ответ сравнивается через точное совпадение.

Низкий общий уровень не означает, что модель бесполезна для проекта. Он показывает, насколько много качества должно прийти от адаптации под русскоязычный VQA-формат.

In [ ]:
by_type_plot_df = metrics_df[metrics_df["metric"].isin(["yes/no", "non-yes/no"])].dropna(subset=["accuracy"]).copy()
by_type_plot_df["Тип ответа"] = by_type_plot_df["metric"].map({
    "yes/no": "да/нет",
    "non-yes/no": "не да/нет",
})
by_type_plot_df["Точность"] = by_type_plot_df["accuracy"]

fig = px.bar(
    by_type_plot_df,
    x="Тип ответа",
    y="Точность",
    color="Тип ответа",
    text=by_type_plot_df["Точность"].map(lambda value: f"{value:.3f}"),
    title="Точность по типу ответа",
    labels={"Тип ответа": "Тип ответа", "Точность": "Точность"},
)
fig.update_yaxes(title_text="Точность", range=[0, 1])
fig.update_xaxes(title_text="Тип ответа")
fig.update_layout(legend_title_text="Тип ответа", showlegend=True)
fig.update_traces(textposition="outside")
fig.show()

Разделение по типу ответа показывает главный дисбаланс базовой проверки. Вопросы `да/нет` модель решает заметно увереннее, потому что пространство ответов маленькое и нормализация объединяет русские и английские варианты.

Остальные вопросы требуют точного объекта, атрибута, материала или пространственного отношения на русском языке. Именно этот класс ошибок должен быть главным фокусом при подготовке train format и LoRA/QLoRA.

In [ ]:
top_error_answers = (
    errors_df["ground_truth_norm"]
    .value_counts()
    .head(10)
    .rename_axis("ground_truth_norm")
    .reset_index(name="count")
)

show_df(top_error_answers)

if len(top_error_answers) > 0:
    top_error_plot_df = top_error_answers.rename(columns={
        "ground_truth_norm": "Правильный ответ",
        "count": "Количество ошибок",
    })

    fig = px.bar(
        top_error_plot_df,
        x="Правильный ответ",
        y="Количество ошибок",
        text="Количество ошибок",
        title="Топ-10 правильных ответов среди ошибок модели",
        labels={
            "Правильный ответ": "Правильный ответ",
            "Количество ошибок": "Количество ошибок",
        },
    )
    fig.update_xaxes(title_text="Правильный ответ")
    fig.update_yaxes(title_text="Количество ошибок")
    fig.update_traces(textposition="outside")
    fig.show()



Частые правильные ответы среди ошибок помогают понять, где базовая модель ошибается системно. В текущем запуске среди ошибок заметны положительные бинарные ответы, пространственные ответы и названия объектов.

Отдельно важны случаи, где модель отвечает английским словом на русский правильный ответ. Для будущего дообучения это сигнал добавить контроль языка ответа, а не ограничиваться только визуальной частью задачи.

## 12. Визуальные примеры

Покажем 10 примеров с картинкой, вопросом, правильным ответом и предсказанием модели. Лучше смотреть вместе верные и неверные ответы: так проще понять, где модель справляется, а где ломается.

In [ ]:
def build_visual_examples(results, max_examples=10):
    correct_part = results[results["is_correct"]].head(max_examples // 2)
    error_part = results[~results["is_correct"]].head(max_examples // 2)

    examples = pd.concat([correct_part, error_part], ignore_index=True)

    if len(examples) < max_examples:
        used_ids = set(examples["row_id"].tolist())
        extra = results[~results["row_id"].isin(used_ids)].head(max_examples - len(examples))
        examples = pd.concat([examples, extra], ignore_index=True)

    return examples.head(max_examples)


def short_text(text, max_len=70):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len] + "..."


visual_examples = build_visual_examples(results_df, max_examples=10).copy()
visual_examples["status"] = np.where(visual_examples["is_correct"], "верно", "неверно")
visual_table = visual_examples[[
    "row_id",
    "status",
    "question",
    "ground_truth",
    "prediction",
]]
show_df(visual_table)

cols = 2
rows = int(np.ceil(len(visual_examples) / cols))
subplot_titles = [
    (
        f"#{idx + 1}: {row['status']}<br>"
        f"Вопрос: {short_text(row['question'])}<br>"
        f"Правильный ответ: {short_text(row['ground_truth'])} | "
        f"Предсказание: {short_text(row['prediction'])}"
    )
    for idx, row in visual_examples.reset_index(drop=True).iterrows()
]

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.04,
    vertical_spacing=0.13,
)

for idx, (_, row) in enumerate(visual_examples.iterrows()):
    image = to_pil_image(row["_image"])
    image.thumbnail((420, 420))
    fig.add_trace(
        go.Image(z=np.array(image)),
        row=idx // cols + 1,
        col=idx % cols + 1,
    )

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_annotations(font_size=11)
fig.update_layout(
    title="Примеры предсказаний модели",
    height=max(520 * rows, 520),
    width=1000,
    showlegend=False,
)
fig.show()

Визуальные примеры показывают, что правильные ответы чаще встречаются на простых атрибутах и бинарных вопросах. Ошибки выглядят разнообразнее: встречаются неверные объекты, неверные пространственные отношения и ответы на английском языке.

Такой ручной просмотр нужен вместе с метриками. Он помогает понять, какие ошибки являются проблемой визуального распознавания, а какие связаны с форматом короткого русскоязычного ответа.

## 13. Итоги базовой проверки

Таблица метрик и графики показывают, что SmolVLM без дообучения решает GQA-ru неравномерно. Бинарные вопросы оказываются существенно проще, а открытые короткие ответы дают основную часть ошибок.

Типичные проблемы базовой проверки: ответы на английском языке, путаница близких объектов, ошибки в пространственных отношениях и нестабильность на положительных вопросах `да/нет`. Эти ошибки важно зафиксировать до LoRA/QLoRA, чтобы потом сравнивать не только общий показатель точного совпадения, но и качество по типам вопросов.

Следующие шаги для проекта: подготовить train format, запустить LoRA/QLoRA дообучение и повторить оценку на той же подвыборке. После проверки пайплайна базовую проверку можно расширить до полного `testdev`.